# Does plot improve revenue prediction after controlling for film metadata?

Compare the same later releases with and without plot features. The primary target is log recorded revenue. The CPI sensitivity uses a common nominal USD assumption. All comparisons use films with a listed production company and CPI coverage.

In [ ]:
import json
import os

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import NMF
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from corpus import CLEAN_PATH

USE_EMBEDDINGS = os.getenv("MOVIE_SENTENCE_EMBEDDINGS") == "1"
if USE_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer


In [ ]:
FOLDS = [(2005, 2008), (2009, 2011), (2012, 2013), (2014, 2016)]
FINAL_WINDOW = "2014–2016"
TARGETS = ["revenue", "revenue_cpi_2016"]
PRODUCTION_FIELDS = ["production_countries", "production_companies"]
GENRES = ["Drama", "Comedy", "Thriller", "Action", "Romance"]
TOKEN_PATTERN = r"(?u)\b[^\W\d_]{2,}\b"

movies = pd.read_csv(CLEAN_PATH)
labeled = movies.dropna(subset=["revenue", "release_year"]).copy()
labeled["normalized_plot"] = labeled["overview"].str.casefold()

for field in ["genres", *PRODUCTION_FIELDS]:
    labeled[field] = labeled[field].fillna("[]").map(json.loads)

# Both targets use the same films for a paired comparison.
has_company = labeled["production_companies"].map(bool)
has_cpi = labeled["revenue_cpi_2016"].notna()
cohort = labeled.loc[has_company & has_cpi]

BOOTSTRAP_DRAWS = 2000
MIN_SLICE_FILMS = 30
FIGURE_PATH = Path("../outputs/paired_plot_errors.png")

## Optional frozen contextual embeddings

Enable with `MOVIE_SENTENCE_EMBEDDINGS=1` and install the optional encoder as described in the README. Encode intact overviews with the checkpoint's tokenizer. For long overviews, split the model's token IDs into windows and average normalized window embeddings with token-count weights.

In [ ]:
embeddings = None

if USE_EMBEDDINGS:
    encoder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
        device="cpu",
    )
    tokenizer = encoder.tokenizer
    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
    window_size = encoder.max_seq_length - special_tokens

    chunks = []
    owners = []
    weights = []

    for row, overview in enumerate(labeled["overview"]):
        tokens = tokenizer.encode(overview, add_special_tokens=False, verbose=False)
        if len(tokens) <= window_size:
            chunks.append(overview)
            owners.append(row)
            weights.append(max(1, len(tokens)))
            continue

        for start in range(0, len(tokens), window_size):
            window = tokens[start:start + window_size]
            chunks.append(tokenizer.decode(window))
            owners.append(row)
            weights.append(len(window))

In [ ]:
if USE_EMBEDDINGS:
    vectors = encoder.encode(
        chunks,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Normalizing the weighted sum gives the same direction as the weighted mean.
    pooled = np.zeros((len(labeled), vectors.shape[1]))
    weighted_vectors = vectors * np.asarray(weights)[:, None]
    np.add.at(pooled, owners, weighted_vectors)

    norms = np.linalg.norm(pooled, axis=1, keepdims=True)
    pooled /= np.maximum(norms, 1e-12)
    embeddings = pd.DataFrame(pooled, index=labeled.index)

    print(f"Encoded {len(embeddings):,} films with {len(chunks) - len(labeled)} extra windows.")

## Descriptive topics

Fit five topics on the broad text corpus for interpretation. Prediction below fits a separate topic model inside each fold.

In [ ]:
descriptive_vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    stop_words="english",
    min_df=2,
)
corpus_words = descriptive_vectorizer.fit_transform(movies["overview"])
terms = descriptive_vectorizer.get_feature_names_out()

topic_model = NMF(n_components=5, init="nndsvda", random_state=42, max_iter=500)
topic_weights = topic_model.fit_transform(corpus_words)
component_norms = np.linalg.norm(topic_model.components_, axis=1)
dominant_topics = (topic_weights * component_norms).argmax(axis=1)
has_terms = corpus_words.getnnz(axis=1) > 0

topic_rows = []
for topic, component in enumerate(topic_model.components_):
    top_terms = component.argsort()[-10:][::-1]
    assigned_films = (dominant_topics == topic) & has_terms

    topic_rows.append({
        "Topic": topic + 1,
        "Top words": ", ".join(terms[top_terms]),
        "Films": assigned_films.sum(),
    })

print(f"{len(terms):,} terms. {(~has_terms).sum():,} overviews have no retained terms.")
topic_summary = pd.DataFrame(topic_rows).set_index("Topic")
topic_summary

### Topic examples and stability

For each topic, show the closest overview among its assigned films. Compare the reference fit with two seeded fits that perturb the initialization. Match topics by cosine similarity, then compare the ten highest-weight terms.

In [ ]:
unit_components = topic_model.components_ / component_norms[:, None]
examples = []

for topic, component in enumerate(unit_components):
    assigned = np.flatnonzero((dominant_topics == topic) & has_terms)
    similarity = corpus_words[assigned] @ component
    example = movies.iloc[assigned[similarity.argmax()]]

    examples.append({
        "Topic": topic + 1,
        "Movie ID": example["movie_id"],
        "Representative overview": example["overview"][:180],
    })

pd.DataFrame(examples).set_index("Topic")

In [ ]:
stability_rows = []
for seed in (17, 23):
    repeated_model = NMF(
        n_components=5, init="nndsvdar", random_state=seed, max_iter=700
    )
    repeated_model.fit(corpus_words)
    repeated = repeated_model.components_
    repeated_norms = np.linalg.norm(repeated, axis=1)
    similarity = unit_components @ (repeated / repeated_norms[:, None]).T
    reference_ids, repeated_ids = linear_sum_assignment(similarity, maximize=True)

    for reference, match in zip(reference_ids, repeated_ids):
        reference_terms = set(unit_components[reference].argsort()[-10:])
        matched_terms = set(repeated[match].argsort()[-10:])
        stability_rows.append({
            "Seed": seed,
            "Topic": reference + 1,
            "Cosine": similarity[reference, match],
            "Top 10 shared terms": len(reference_terms & matched_terms),
            "Iterations": repeated_model.n_iter_,
        })

pd.DataFrame(stability_rows).set_index(["Seed", "Topic"]).round(3)

## Training-fold features

Controls are release year and month, genre, runtime, collection membership, and production company. Collection membership is a franchise proxy. Company indicators are multi-label. A company needs five earlier training films to get its own indicator. Other companies share a rare indicator.

In [ ]:
def indicators(
    train: pd.DataFrame,
    test: pd.DataFrame,
    fields: list[str],
) -> tuple[csr_matrix, csr_matrix]:
    """Fit category indicators using training data.

    Args:
        train (pd.DataFrame): Training films.
        test (pd.DataFrame): Evaluation films.
        fields (list[str]): Columns containing category lists.

    Returns:
        tuple[csr_matrix, csr_matrix]: Training and evaluation indicators.
    """
    known_companies = set()
    if "production_companies" in fields:
        frequencies = train["production_companies"].explode().value_counts()
        known_companies = set(frequencies[frequencies.ge(5)].index)

    records = []
    for frame in (train, test):
        frame_records = []
        for row in frame[fields].itertuples(index=False, name=None):
            indicators_for_film = {}
            for field, values in zip(fields, row):
                for value in values:
                    if field == "production_companies" and value not in known_companies:
                        value = "rare"
                    indicators_for_film[f"{field}={value}"] = 1
            frame_records.append(indicators_for_film)
        records.append(frame_records)

    train_records, test_records = records
    vectorizer = DictVectorizer()
    return vectorizer.fit_transform(train_records), vectorizer.transform(test_records)

In [ ]:
def control_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> dict[str, tuple[csr_matrix, csr_matrix]]:
    """Build named control blocks from earlier films.

    Args:
        train (pd.DataFrame): Training films.
        test (pd.DataFrame): Evaluation films.

    Returns:
        dict[str, tuple[csr_matrix, csr_matrix]]: Train/test matrices by block.
    """
    months = np.eye(12)
    year_scaler = StandardScaler()
    train_year = year_scaler.fit_transform(train[["release_year"]])
    test_year = year_scaler.transform(test[["release_year"]])

    train_month = months[train["release_month"].astype(int) - 1]
    test_month = months[test["release_month"].astype(int) - 1]
    time = (
        hstack([csr_matrix(train_year), csr_matrix(train_month)], format="csr"),
        hstack([csr_matrix(test_year), csr_matrix(test_month)], format="csr"),
    )

    runtime_scaler = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    train_runtime = runtime_scaler.fit_transform(train[["runtime"]])
    test_runtime = runtime_scaler.transform(test[["runtime"]])
    runtime = (csr_matrix(train_runtime), csr_matrix(test_runtime))

    franchise = (
        csr_matrix(train[["collection_id"]].notna().to_numpy(dtype=float)),
        csr_matrix(test[["collection_id"]].notna().to_numpy(dtype=float)),
    )
    return {
        "Release time": time,
        "Genre": indicators(train, test, ["genres"]),
        "Runtime": runtime,
        "Franchise": franchise,
        "Production company": indicators(train, test, ["production_companies"]),
    }

In [ ]:
def combine_features(
    *pairs: tuple[csr_matrix, csr_matrix],
) -> tuple[csr_matrix, csr_matrix]:
    """Stack matching training and evaluation feature blocks.

    Args:
        *pairs (tuple[csr_matrix, csr_matrix]): Train and test blocks in order.

    Returns:
        tuple[csr_matrix, csr_matrix]: Combined train and test matrices.
    """
    train_blocks, test_blocks = zip(*pairs)
    return hstack(train_blocks, format="csr"), hstack(test_blocks, format="csr")

In [ ]:
def plot_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[dict[str, tuple[csr_matrix, csr_matrix]], int]:
    """Encode overviews with training-fold vocabulary and topics.

    Args:
        train (pd.DataFrame): Training films.
        test (pd.DataFrame): Evaluation films.

    Returns:
        tuple[dict, int]: Feature blocks and the fitted vocabulary size.
    """
    vectorizer = TfidfVectorizer(
        token_pattern=TOKEN_PATTERN,
        stop_words="english",
        min_df=2,
    )
    train_words = vectorizer.fit_transform(train["overview"])
    test_words = vectorizer.transform(test["overview"])
    features = {"TF-IDF": (train_words, test_words)}

    nmf = NMF(n_components=5, init="nndsvda", random_state=42, max_iter=500)
    train_topics = nmf.fit_transform(train_words)
    test_topics = nmf.transform(test_words)
    features["NMF"] = (csr_matrix(train_topics), csr_matrix(test_topics))

    if embeddings is not None:
        train_vectors = embeddings.loc[train.index].to_numpy()
        test_vectors = embeddings.loc[test.index].to_numpy()
        features["Sentence embedding"] = (
            csr_matrix(train_vectors),
            csr_matrix(test_vectors),
        )

    return features, len(vectorizer.vocabulary_)

## Compare on the same later films

Fit the full controls model and add each plot representation in turn. Separately drop one control at a time from the controls-only model. All regressions use the same fixed ridge penalty. Plot comparisons always include time. Company support is calculated separately for each training fold.

In [ ]:
fold_errors = []
fold_counts = []

for start, end in FOLDS:
    window = f"{start}–{end}"
    test = cohort.loc[cohort["release_year"].between(start, end)]
    earlier = cohort["release_year"].lt(start)
    duplicate = cohort["normalized_plot"].isin(test["normalized_plot"])
    train = cohort.loc[earlier & ~duplicate]

    controls = control_features(train, test)
    plots, vocabulary_size = plot_features(train, test)
    full = combine_features(*controls.values())

    models = {"Controls": full}
    for name, features in plots.items():
        models[f"Plot: {name}"] = combine_features(full, features)

    for omitted in controls:
        retained = [features for name, features in controls.items() if name != omitted]
        models[f"Omit: {omitted}"] = combine_features(*retained)

    company_counts = train["production_companies"].explode().value_counts()
    supported_companies = set(company_counts[company_counts.ge(5)].index)
    has_supported_company = test["production_companies"].map(
        lambda companies: bool(supported_companies.intersection(companies))
    )

    for target in TARGETS:
        y_train = np.log(train[target].to_numpy(dtype=float))
        y_test = np.log(test[target].to_numpy(dtype=float))

        errors = test[["movie_id", "release_year", "genres", "overview"]].copy()
        errors["Window"] = window
        errors["Target"] = target
        errors["Company supported in training"] = has_supported_company

        for name, (train_features, test_features) in models.items():
            ridge = Ridge(alpha=1.0, solver="lsqr")
            ridge.fit(train_features, y_train)
            residual = ridge.predict(test_features) - y_test
            errors[name] = np.abs(residual)

        fold_errors.append(errors)

    fold_counts.append({
        "Window": window,
        "Training films": len(train),
        "Evaluation films": len(test),
        "Vocabulary": vocabulary_size,
    })

errors = pd.concat(fold_errors, ignore_index=True)
pd.DataFrame(fold_counts).set_index("Window")

## Paired log-MAE changes

Negative plot changes favor adding plot. Positive omission changes mean that dropping the control worsened prediction. The 95% percentile intervals resample films within release year and are conditional on the observed years and fitted models. They are pointwise, and overlapping slices are descriptive.

In [ ]:
def paired_summary(
    films: pd.DataFrame,
    comparisons: list[str],
) -> pd.DataFrame:
    """Summarize paired MAE changes with a year-stratified film bootstrap.

    Args:
        films (pd.DataFrame): Films with absolute errors for the same models.
        comparisons (list[str]): Alternative model columns to compare with Controls.

    Returns:
        pd.DataFrame: Sample size, MAEs, paired differences, and interval endpoints.
    """
    deltas = films[comparisons].sub(films["Controls"], axis=0)
    rng = np.random.default_rng(42)
    bootstrap = np.zeros((BOOTSTRAP_DRAWS, len(comparisons)))

    for _, year in deltas.groupby(films["release_year"]):
        values = year.to_numpy()
        sampled = rng.integers(len(values), size=(BOOTSTRAP_DRAWS, len(values)))
        bootstrap += values[sampled].sum(axis=1)

    lower, upper = np.quantile(bootstrap / len(films), [0.025, 0.975], axis=0)
    return pd.DataFrame({
        "Films": len(films),
        "Controls MAE": films["Controls"].mean(),
        "Comparison MAE": films[comparisons].mean(),
        "Δ MAE": deltas.mean(),
        "Lower 95%": lower,
        "Upper 95%": upper,
    })

In [ ]:
plot_models = errors.columns[errors.columns.str.startswith("Plot: ")].tolist()
control_models = errors.columns[errors.columns.str.startswith("Omit: ")].tolist()
recorded = errors.loc[errors["Target"].eq("revenue")]

plot_tables = {}
for target, target_films in errors.groupby("Target", sort=False):
    by_window = {"Pooled": paired_summary(target_films, plot_models)}
    for window, films in target_films.groupby("Window", sort=False):
        by_window[window] = paired_summary(films, plot_models)
    plot_tables[target] = pd.concat(by_window, names=["Window", "Model"])

plot_results = pd.concat(plot_tables, names=["Target"])
display(plot_results.loc["revenue"].round(3))
display(plot_results.loc["revenue_cpi_2016"].round(3))

In [ ]:
control_tables = {"Pooled": paired_summary(recorded, control_models)}
for window, films in recorded.groupby("Window", sort=False):
    control_tables[window] = paired_summary(films, control_models)

control_results = pd.concat(control_tables, names=["Window", "Model"])
control_results.round(3)

## Where does plot help?

Show major genres overall and by release window. For production, compare films with and without at least one company represented in training. Each displayed group needs at least 30 distinct films. A genre can overlap other genres.

In [ ]:
primary_plot = "Plot: Sentence embedding" if embeddings is not None else "Plot: TF-IDF"
genre_films = recorded.explode("genres")
genre_films = genre_films.loc[genre_films["genres"].isin(GENRES)]

genre_results = {}
for genre, films in genre_films.groupby("genres"):
    genre_results[("Pooled", genre)] = paired_summary(films, [primary_plot]).iloc[0]

for key, films in genre_films.groupby(["Window", "genres"]):
    if films["movie_id"].nunique() >= MIN_SLICE_FILMS:
        genre_results[key] = paired_summary(films, [primary_plot]).iloc[0]

genre_summary = pd.DataFrame.from_dict(genre_results, orient="index")
genre_summary.index.names = ["Window", "Genre"]
genre_summary["Films"] = genre_summary["Films"].astype(int)
genre_summary.round(3)

In [ ]:
company_results = {}
for supported, films in recorded.groupby("Company supported in training"):
    label = "Represented" if supported else "Rare or unseen"
    company_results[("Pooled", label)] = paired_summary(films, [primary_plot]).iloc[0]

for (window, supported), films in recorded.groupby(
    ["Window", "Company supported in training"]
):
    if len(films) >= MIN_SLICE_FILMS:
        label = "Represented" if supported else "Rare or unseen"
        company_results[(window, label)] = paired_summary(films, [primary_plot]).iloc[0]

company_summary = pd.DataFrame.from_dict(company_results, orient="index")
company_summary.index.names = ["Window", "Company support"]
company_summary["Films"] = company_summary["Films"].astype(int)
company_summary.round(3)

In [ ]:
figures = plot_results.loc["revenue"].xs(primary_plot, level="Model")

with plt.rc_context({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "figure.dpi": 150,
    "savefig.dpi": 300,
}):
    fig, axes = plt.subplots(1, 2, figsize=(7, 3), sharex=True, layout="constrained")
    representation = primary_plot.removeprefix("Plot: ")
    fig.suptitle(f"{representation} added to full controls")

    panels = [
        (figures, "(a) Release window"),
        (genre_summary.loc["Pooled"], "(b) Major genre, pooled"),
    ]

    for ax, (summary, title) in zip(axes, panels):
        labels = [f"{name}  (n={row['Films']:,.0f})" for name, row in summary.iterrows()]
        positions = np.arange(len(summary))
        ax.hlines(positions, summary["Lower 95%"], summary["Upper 95%"], color="#245778")
        ax.scatter(summary["Δ MAE"], positions, s=22, color="#245778", zorder=3)
        ax.axvline(0, color="#737373", linewidth=0.9, linestyle="--")
        ax.grid(axis="x", color="#dddddd", linewidth=0.5)
        ax.set(yticks=positions, yticklabels=labels, xlabel="Change in log MAE", title=title)
        ax.invert_yaxis()

    FIGURE_PATH.parent.mkdir(exist_ok=True)
    fig.savefig(FIGURE_PATH)
    plt.show()

### Film-level examples

Inspect the largest improvements and deteriorations in the latest window. These are examples of model behavior. Plot text alone cannot establish why the prediction changed.

In [ ]:
final_films = recorded.loc[recorded["Window"].eq(FINAL_WINDOW)].copy()
final_films["Δ Absolute error"] = final_films[primary_plot] - final_films["Controls"]

helped = final_films.nsmallest(3, "Δ Absolute error")
hurt = final_films.nlargest(3, "Δ Absolute error")
examples = pd.concat({"Helped": helped, "Hurt": hurt}, names=["Direction"])
examples["Overview excerpt"] = examples["overview"].str.slice(0, 180)

fields = ["movie_id", "Controls", primary_plot, "Δ Absolute error", "Overview excerpt"]
examples[fields].reset_index(level=1, drop=True).round(3)